In [100]:
import cv2
import numpy as np
import sys
import os
 
# ═════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ═════════════════════════════════════════════════════════════════════════════
SYRINGE_ML   = 10.0
SCALE        = 0.33    # run at 1/3 resolution → ~10ms per frame
MARK_DARK    = 85      # gray threshold for graduation marks
PLUNGER_DARK = 65      # gray threshold for plunger rubber
# ═════════════════════════════════════════════════════════════════════════════
 
IMAGE_PATH  = "inj3.png"
OUTPUT_PATH = "result5prob.png"
 
FONT      = cv2.FONT_HERSHEY_SIMPLEX
FONT_BOLD = cv2.FONT_HERSHEY_DUPLEX

In [101]:
def detect(gray):
    """
    Fast, fully automatic syringe detection.

    Speed  : ~10 ms on 1320×1760 input (all processing on 1/3-res image).
    0ml fix: barrel top is located first; graduation mark scan only runs
             BELOW that line, so needle/nozzle above the cylinder can
             never be mislabelled as 0 ml.
    11ml fix: marks are hard-capped at 11 entries (0–10 ml only).
    """
    h, w   = gray.shape
    small  = cv2.resize(gray, (int(w * SCALE), int(h * SCALE)))
    sh, sw = small.shape
    dark   = (small < MARK_DARK).astype(np.int8)

    # ── 1. Find the best scan column: most evenly-spaced thin dark lines ──────
    y0s, y1s = int(sh * 0.20), int(sh * 0.80)
    best_x_s, best_marks_s = 0, []

    for x in range(20, sw - 10):
        col    = dark[y0s:y1s, x]
        tr     = np.diff(col)
        starts = np.where(tr == 1)[0]
        ends   = np.where(tr == -1)[0]
        if len(starts) == 0 or len(ends) == 0: continue
        if ends[0] < starts[0]: ends = ends[1:]
        n = min(len(starts), len(ends))
        starts, ends = starts[:n], ends[:n]
        thin    = (ends - starts >= 1) & (ends - starts <= 5)
        centers = ((starts + ends) // 2 + y0s)[thin]
        if len(centers) < 5: continue
        diffs = np.diff(centers)
        for base in range(13, 24):          # ~20px spacing at 0.33 scale
            ok    = (diffs >= base * 0.75) & (diffs <= base * 1.35)
            group = [centers[0]]
            for i, flag in enumerate(ok):
                if flag: group.append(centers[i + 1])
            if len(group) > len(best_marks_s):
                best_marks_s = group[:]
                best_x_s     = x

    if len(best_marks_s) < 5:
        return 0, [], 0, 0, 0, None

    best_marks_s = [int(m) for m in best_marks_s[:11]]  # hard cap at 11

    # ── 2. Barrel walls: local Sobel around the scan column ──────────────────
    sobelx     = cv2.Sobel(small, cv2.CV_32F, 1, 0, ksize=3)
    lo         = max(0,    best_x_s - 70)
    hi         = min(sw,   best_x_s + 70)
    col_edge   = np.abs(sobelx[int(sh*0.25):int(sh*0.75), lo:hi]).sum(axis=0)
    col_smooth = np.convolve(col_edge, np.ones(3) / 3, mode='same')
    left_s     = int(np.argmax(col_smooth[:len(col_smooth) // 2])) + lo
    tmp        = col_smooth.copy(); tmp[:len(col_smooth) // 2] = 0
    right_s    = int(np.argmax(tmp)) + lo
    if right_s <= left_s: right_s = left_s + 30

    # ── 3. Barrel top: first row with ≥25% dark pixels inside barrel ─────────
    #    All marks ABOVE this row are rejected — prevents needle being 0 ml.
    barrel_top_s = int(sh * 0.12)
    for y in range(int(sh * 0.06), int(sh * 0.55)):
        row = small[y, left_s:right_s]
        if len(row) > 0 and (row < 90).sum() > len(row) * 0.25:
            barrel_top_s = y
            break

    # Remove any marks above the barrel top
    best_marks_s = [m for m in best_marks_s if m >= barrel_top_s][:11]
    if len(best_marks_s) < 5:
        return 0, [], 0, 0, 0, None

    # ── 4. Plunger: dark contour below barrel midpoint ────────────────────────
    blurred    = cv2.GaussianBlur(small, (5, 5), 0)
    _, thresh  = cv2.threshold(blurred, PLUNGER_DARK, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    plunger_s  = None
    for c in sorted(contours, key=cv2.contourArea, reverse=True):
        x, y, cw, ch = cv2.boundingRect(c)
        if (cv2.contourArea(c) > 300 and y > sh * 0.45
                and x < right_s and (x + cw) > left_s and cw > 10):
            plunger_s = (x, y, cw, ch)
            break

    # ── 5. Scale all coords back to original resolution ───────────────────────
    def up(v): return int(v / SCALE)
    return (up(best_x_s),
            [up(m) for m in best_marks_s],
            up(left_s), up(right_s),
            up(barrel_top_s),
            tuple(up(v) for v in plunger_s) if plunger_s else None)







In [102]:
def draw(img, marks, bx1, bx2, plunger, ml):
    out           = img.copy()
    spacing       = float(np.mean(np.diff(marks)))
    zero_ml_y     = marks[0]
    plunger_top_y = plunger[1]
    px, py, pw, ph = plunger
    label_x       = bx2 + 14

    # Liquid fill overlay
    ov = out.copy()
    cv2.rectangle(ov, (bx1 + 2, zero_ml_y), (bx2 - 2, plunger_top_y), (80, 160, 255), -1)
    cv2.addWeighted(ov, 0.18, out, 0.82, 0, out)

    # Graduation marks 0 ml → 10 ml (index = label, capped at 10)
    for i, my in enumerate(marks):
        n = i
        if   n == 0:  c,t,tk,fs = (0,255,100), 3, 55, 0.75
        elif n == 10: c,t,tk,fs = (0,220,255), 3, 55, 0.75
        elif n == 5:  c,t,tk,fs = (255,180, 0), 3, 50, 0.72
        else:         c,t,tk,fs = (255,255, 80), 2, 35, 0.60
        cv2.line(out, (bx1 - tk, my), (bx1 - 2, my), c, t)
        cv2.line(out, (bx1 - 2,  my), (bx2 + 2, my), c, 1)
        cv2.putText(out, f"{n} ml", (label_x, my + 7), FONT, fs, c, t)

    # Plunger box + measurement line
    cv2.rectangle(out, (px, py), (px + pw, py + ph), (0, 230, 60), 3)
    cv2.line(out, (bx1 - 5, plunger_top_y), (bx2 + 5, plunger_top_y), (0, 0, 255), 4)
    cv2.arrowedLine(out, (bx2 + 130, plunger_top_y),
                         (bx2 +  12, plunger_top_y), (0, 0, 255), 3, tipLength=0.25)
    cv2.putText(out, "Plunger top", (bx2 + 140, plunger_top_y - 8),
                FONT, 0.55, (0, 0, 255), 2)

    # Volume readout box
    bx, by, bw, bh = 20, 20, 240, 110
    cv2.rectangle(out, (bx, by), (bx + bw, by + bh), (15, 15, 15), -1)
    cv2.rectangle(out, (bx, by), (bx + bw, by + bh), (0, 220, 220),  3)
    cv2.putText(out, f"{ml} ml",         (bx + 14, by + 68), FONT_BOLD, 2.2, (0,255,255), 5)
    cv2.putText(out, "PLUNGER ESTIMATE", (bx + 14, by + 94), FONT,      0.45, (160,160,160), 1)

    # Fill bar
    bx2b, by2, bh2 = 20, 140, 18
    bar_max  = 240
    bar_fill = int((ml / SYRINGE_ML) * bar_max)
    cv2.rectangle(out, (bx2b, by2), (bx2b + bar_max,  by2 + bh2), (50,  50,  50), -1)
    cv2.rectangle(out, (bx2b, by2), (bx2b + bar_fill, by2 + bh2), (0, 200, 200), -1)
    cv2.putText(out, f"{ml / SYRINGE_ML * 100:.0f}% of {SYRINGE_ML:.0f} ml",
                (bx2b, by2 + bh2 + 22), FONT, 0.55, (160, 160, 160), 1)
    return out


In [103]:

def analyse(image_path, output_path):
    if not os.path.exists(image_path):
        print(f"[ERROR] File not found: '{image_path}'"); return
    img = cv2.imread(image_path)
    if img is None:
        print(f"[ERROR] Cannot read: '{image_path}'"); return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    scan_x, marks, bx1, bx2, btop, plunger = detect(gray)

    if len(marks) < 5:
        print("[ERROR] Not enough graduation marks."); return
    if plunger is None:
        print("[ERROR] Plunger not detected."); return

    spacing       = float(np.mean(np.diff(marks)))
    plunger_top_y = plunger[1]
    zero_y        = marks[0]
    raw_ml        = (plunger_top_y - zero_y) / spacing
    ml            = round(min(SYRINGE_ML, max(0.0, raw_ml)), 1)  # strict 0-10

    print(f"[INFO] {len(marks)} marks | spacing={spacing:.1f}px | barrel=[{bx1},{bx2}] top={btop}")
    print(f"[INFO] 0ml at Y={zero_y} (barrel top Y={btop}) | plunger Y={plunger_top_y}")
    print(f"\n{'═'*44}")
    print(f"  ►  VOLUME = {ml} ml  ({ml/SYRINGE_ML*100:.0f}%)")
    print(f"{'═'*44}\n")

    result = draw(img, marks, bx1, bx2, plunger, ml)
    cv2.imwrite(output_path, result)
    print(f"[DONE] Saved → {output_path}")
    try:
        cv2.imshow("Syringe", cv2.resize(result, (660, 880)))
        cv2.waitKey(0); cv2.destroyAllWindows()
    except cv2.error:
        pass


In [104]:
if __name__ == "__main__":
    clean = [a for a in sys.argv[1:] if not a.startswith("-") and not a.endswith(".json")]
    image_file  = clean[0] if len(clean) > 0 else IMAGE_PATH
    output_file = clean[1] if len(clean) > 1 else OUTPUT_PATH
    print(f"[INFO] Input  : {image_file}")
    print(f"[INFO] Output : {output_file}")
    analyse(image_file, output_file)

[INFO] Input  : inj3.png
[INFO] Output : result5prob.png
[ERROR] Not enough graduation marks.
